In [0]:
%fs ls /Volumes/cinedata/bronze/inputs

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

#caminho para os dados
volume_path = "/Volumes/cinedata/bronze/inputs"

#dicionario de mapeamento das tabelas csv -> db de acordo com o pdf da atividade
tabelas_mapeamento = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

#loop for de leitura e transformacao dos arquivos csv em tabela delta sem alterar o tipo dos dados
for arquivo, nome_tabela in tabelas_mapeamento.items():
    df = spark.read.csv(f"{volume_path}/{arquivo}", header=True, inferSchema=True)
    df = df.withColumn("ingestion_datetime", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(nome_tabela)                  #gravação das tabelas em formato delta usando append
    print(f"Arquivo {arquivo} transformado em tabela {nome_tabela} com sucesso!")

In [0]:
%sql
SELECT * FROM bronze.tb_movies_info LIMIT 10; 

In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp

#aplicando a sugestão de verificação da ultima semana de dados para utilização dos widgets do databricks (os parametros)
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)          

data_fim_padrao = hoje.strftime("%m-%d-%Y")                     #string da data de hoje
data_inicio_padrao = sete_dias_atras.strftime("%m-%d-%Y")       #string da data de sete dias atrás

#criação dos widgets
dbutils.widgets.text("data_inicio", data_inicio_padrao, "Data Inicio (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_padrao, "Data Fim (MM-DD-AAAA)")

#aplicando valores aos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

#captura do endpoint da api
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

#requisição
resposta = requests.get(url)

#transformação da resposta em json
dados = resposta.json()

#extrair os dados relevantes
cotacoes = dados["value"]

#condicional para verificação da existencia de cotações no periodo
if cotacoes:
    df_cotacao = spark.createDataFrame(cotacoes)   #cria o dataframe cotações
    df_cotacao = df_cotacao.withColumn("ingestion_datetime", current_timestamp())   #adiciona a coluna de ingestão novamente para possivel consulta
    df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")   #grava como tabela delta usando append 
    print("Sucesso na ingestão da cotação do dólar!")
else:
    print("Não foram encontradas cotações no período especificado")


In [0]:
%sql
SELECT * FROM bronze.tb_cotacao_dolar LIMIT 10;